In [11]:
from langchain_classic.vectorstores import FAISS
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
llm = ChatGroq(model='qwen/qwen3.8-27b', max_tokens=300)

In [6]:
embedding_model = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001', output_dimensionality=100)

In [8]:
docs = [
    Document(
        page_content="""Python is widely used for machine learning and data analysis because of libraries like Pandas and Scikit-learn. The human heart pumps oxygen-rich blood throughout the body to maintain proper organ functioning. Solar panels convert sunlight into electricity and help reduce dependence on fossil fuels. LangChain provides tools like retrievers, embeddings, and vector stores to build RAG applications.""",
        metadata={"source": "Doc1"}
    ),
    Document(
        page_content="""Electric vehicles use rechargeable batteries to power electric motors and reduce carbon emissions. The Amazon rainforest produces a significant amount of the world's oxygen and supports millions of plant and animal species. SQL is used to retrieve, filter, and manipulate structured data stored in relational databases. Regular exercise improves cardiovascular health, strengthens muscles, and helps maintain mental well-being.""",
        metadata={"source": "Doc1"}
    )
]

In [9]:
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [10]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k":5})

In [12]:
compressor = LLMChainExtractor.from_llm(llm)

In [13]:
compression_retriver = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [22]:
query = "what human heart do & how to improve cardiovascular health?"
results = compression_retriver.invoke(query)

In [23]:
for i, doc in enumerate(results):
    print(f"-- Result {i+1} --")
    print(doc.page_content)

-- Result 1 --
The human heart pumps oxygen-rich blood throughout the body to maintain proper organ functioning.
-- Result 2 --
Regular exercise improves cardiovascular health, strengthens muscles, and helps maintain mental well-being.
